# Finding Hamiltonian Ground State Energy

A new function was added to the BlueQubit Python SDK: `bluequbit.library.hamiltonian_gs(qubit_op, options=None, progress_queue=None)`. It calculates the ground state of the Hamiltonian provided in Pauli Sum form (`qubit_op`) using the **Variational Quantum Eigensolver (VQE)** method. The function is available on BlueQubit [hybrid Jupyter notebooks only](https://app.bluequbit.io/hybrid-jobs-notebook) for Premium tier subscription users.

This tutorial demonstrates how to find the **ground state energy** of the hydrogen molecule (H₂) using the new function. The problem definition and solve flow follows the book *A Practical Guide to Quantum Machine Learning and Quantum Optimization* (Combarro, González-Castillo, Di Meglio, 2023).

## Background

Finding the ground state energy of a molecule is a fundamental problem in quantum chemistry — it determines the molecule's stability and chemical properties. For H₂, this means finding the lowest eigenvalue of its Hamiltonian operator.

The workflow follows three stages:

1. **Fermionic Hamiltonian** — Use classical chemistry software (PySCF) to compute how electrons occupy the molecular orbitals of H₂ via *second quantization*.
2. **Qubit mapping** — Transform the fermionic creation/annihilation operators into qubit Pauli operators using the *Jordan-Wigner transformation*.
3. **VQE** — Apply the *Variational Quantum Eigensolver* to find the minimum expectation value (ground state energy) of the qubit Hamiltonian.

BlueQubit's `hamiltonian_gs()` handles step 3 entirely — it sets up the parameterized ansatz circuit, runs the classical optimizer, and optionally offloads circuit execution to BlueQubit's platform cloud.

## 0. Install Required Packages

`bluequbit` is pre-installed in BlueQubit hybrid notebooks. Run the cell below to install the quantum chemistry dependency needed to build the H₂ Hamiltonian.

In [ ]:
!persist_pip qiskit-nature[pyscf]==0.7.2 qiskit==1.4.2 numpy==1.26.4 qiskit-algorithms==0.3.1
# need to restart the kernel after this cell

/workspaces/sdk/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 1. Build the Qubit Hamiltonian

To run VQE we need a **qubit Hamiltonian** — the molecule's energy operator expressed as a sum of Pauli tensor products (e.g. `0.171 * IIIZ + 0.045 * XXXX + ...`).

### Step 1a: Fermionic Hamiltonian via PySCF

We place two hydrogen atoms at coordinates (0, 0, ±0.37 Å), which is close to the H₂ equilibrium bond length of 0.74 Å. PySCF computes the *fermionic Hamiltonian* via second quantization — a description of how electrons move between the four available spin orbitals using creation (`+`) and annihilation (`-`) operators.

### Step 1b: Jordan-Wigner Mapping → Qubit Hamiltonian

The Jordan-Wigner transformation maps each spin orbital to one qubit, converting fermionic operators into strings of Pauli matrices. Our 4-orbital system maps to exactly **4 qubits**, producing a Hamiltonian with 15 Pauli terms that a quantum computer can evaluate.

In [2]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Define H₂ near equilibrium geometry: bond length ~0.74 Å (coordinates in Ångströms)
driver = PySCFDriver(atom="H 0.0 0.0 -0.37; H 0.0 0.0 0.37", basis="sto3g")
problem = driver.run()

# Compute the fermionic Hamiltonian via second quantization
hamiltonian = problem.hamiltonian.second_q_op()
print("Fermionic Hamiltonian (36 terms over 4 spin orbitals):")
print(hamiltonian)

# Apply Jordan-Wigner transformation: 4 orbitals → 4 qubits
mapper = JordanWignerMapper()
qubit_op = mapper.map(hamiltonian)
print("\nQubit Hamiltonian (15 Pauli terms on 4 qubits):")
print(qubit_op)

Fermionic Hamiltonian (36 terms over 4 spin orbitals):
Fermionic Operator
number spin orbitals=4, number terms=36
  0.33737796340722415 * ( +_0 +_0 -_0 -_0 )
+ 0.3318557006754068 * ( +_0 +_1 -_1 -_0 )
+ 0.33737796340722415 * ( +_0 +_2 -_2 -_0 )
+ 0.3318557006754068 * ( +_0 +_3 -_3 -_0 )
+ 0.09060523100759847 * ( +_0 +_0 -_1 -_1 )
+ 0.09060523100759847 * ( +_0 +_1 -_0 -_1 )
+ 0.09060523100759847 * ( +_0 +_2 -_3 -_1 )
+ 0.09060523100759847 * ( +_0 +_3 -_2 -_1 )
+ 0.09060523100759847 * ( +_1 +_0 -_1 -_0 )
+ 0.09060523100759847 * ( +_1 +_1 -_0 -_0 )
+ 0.09060523100759847 * ( +_1 +_2 -_3 -_0 )
+ 0.09060523100759847 * ( +_1 +_3 -_2 -_0 )
+ 0.3318557006754068 * ( +_1 +_0 -_0 -_1 )
+ 0.3488257522452317 * ( +_1 +_1 -_1 -_1 )
+ 0.3318557006754068 * ( +_1 +_2 -_2 -_1 )
+ 0.3488257522452317 * ( +_1 +_3 -_3 -_1 )
+ 0.33737796340722415 * ( +_2 +_0 -_0 -_2 )
+ 0.3318557006754068 * ( +_2 +_1 -_1 -_2 )
+ 0.33737796340722415 * ( +_2 +_2 -_2 -_2 )
+ 0.3318557006754068 * ( +_2 +_3 -_3 -_2 )
+ 0.0906052310

The qubit Hamiltonian is a weighted sum of 15 Pauli tensor products on 4 qubits. A few terms worth noting:

- `−0.812 * IIII` — a constant energy offset (nuclear repulsion + core energy contributions)
- `0.045 * XXXX`, `0.045 * YYYY`, etc. — the off-diagonal terms that create quantum correlations between electrons; these cannot be captured by classical mean-field methods and are what makes this problem interesting for quantum computers
- `0.171 * IIIZ` — a single-qubit Z term encoding one-body orbital energies

## 2. Run VQE with BlueQubit's `hamiltonian_gs()`

`hamiltonian_gs(qubit_op, options=...)` implements VQE under the hood:

1. **Ansatz** — Builds an `EfficientSU2` parameterized circuit with R_Y/R_Z rotation layers and CNOT entangling gates. With `reps=1` and linear entanglement on 4 qubits, this gives 16 tunable parameters.
2. **Optimizer** — A classical optimizer (default: COBYLA) iteratively adjusts the parameters to minimize the energy expectation value.
3. **Estimator** — Evaluates `⟨ψ(θ)|H|ψ(θ)⟩` on each iteration using either a local Aer simulator or BlueQubit's cloud hardware.

**Key `options` keys:**

| Key | Default | Description |
|---|---|---|
| `"reps"` | `1` | Ansatz depth — more reps = more expressive but harder to optimize |
| `"seed"` | `None` | Random seed for reproducibility |
| `"optimizer"` | `"COBYLA"` | Classical optimizer: `"COBYLA"`, `"SLSQP"`, or `"L_BFGS_B"` |
| `"estimator"` | `"qiskit_aer"` | `"qiskit_aer"` for local simulation, `"bluequbit"` for cloud hardware |

The function returns a `VQEResult` with `optimal_value` (ground state energy), `optimal_parameters`, and `cost_function_evals`.

In [3]:
from bluequbit.library.hamiltonian_gs import hamiltonian_gs

# Run VQE with reps=1: the EfficientSU2 default — a shallow ansatz that works well for H₂.
# Energy is logged every 10 iterations so you can watch convergence.
result = hamiltonian_gs(qubit_op, options={"reps": 1, "seed": 1234, "optimizer": "COBYLA"})

# Store energy now so the verification cell below can use it directly
vqe_energy = result.optimal_value.real

print(f"Ground state energy (VQE):    {vqe_energy:.6f} Hartree")
print(f"Cost function evaluations:    {result.cost_function_evals}")
print(f"Number of circuit parameters: {len(result.optimal_parameters)}")

[BQ-PYTHON-SDK][INFO] - Iteration 10, Energy -0.9482364471143615
[BQ-PYTHON-SDK][INFO] - Iteration 20, Energy -1.1153861061180168
[BQ-PYTHON-SDK][INFO] - Iteration 30, Energy -1.137999383761244
[BQ-PYTHON-SDK][INFO] - Iteration 40, Energy -1.1543229919676725
[BQ-PYTHON-SDK][INFO] - Iteration 50, Energy -1.1731662289340774
[BQ-PYTHON-SDK][INFO] - Iteration 60, Energy -1.2443311793731202
[BQ-PYTHON-SDK][INFO] - Iteration 70, Energy -1.4196644051301417
[BQ-PYTHON-SDK][INFO] - Iteration 80, Energy -1.4553102617209432
[BQ-PYTHON-SDK][INFO] - Iteration 90, Energy -1.6994114224342078
[BQ-PYTHON-SDK][INFO] - Iteration 100, Energy -1.7398095622269982
[BQ-PYTHON-SDK][INFO] - Iteration 110, Energy -1.7946799145820522
[BQ-PYTHON-SDK][INFO] - Iteration 120, Energy -1.7948112795269613
[BQ-PYTHON-SDK][INFO] - Iteration 130, Energy -1.8197195648551097
[BQ-PYTHON-SDK][INFO] - Iteration 140, Energy -1.8179325272068552
[BQ-PYTHON-SDK][INFO] - Iteration 150, Energy -1.8266677000801186
[BQ-PYTHON-SDK][INFO

/tmp/ipykernel_40468/4112466131.py:5: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  result = hamiltonian_gs(


[BQ-PYTHON-SDK][INFO] - Iteration 190, Energy -1.8308474736381526
[BQ-PYTHON-SDK][INFO] - Iteration 200, Energy -1.8314596949791797
[BQ-PYTHON-SDK][INFO] - Iteration 210, Energy -1.831369804036309
[BQ-PYTHON-SDK][INFO] - Iteration 220, Energy -1.8316717316237723
[BQ-PYTHON-SDK][INFO] - Iteration 230, Energy -1.8317645566726892
[BQ-PYTHON-SDK][INFO] - Iteration 240, Energy -1.831811413987759
[BQ-PYTHON-SDK][INFO] - Iteration 250, Energy -1.8318429852493896
[BQ-PYTHON-SDK][INFO] - Iteration 260, Energy -1.831852668321266
[BQ-PYTHON-SDK][INFO] - Iteration 270, Energy -1.8318592523565542
[BQ-PYTHON-SDK][INFO] - Iteration 280, Energy -1.8318601072621317
[BQ-PYTHON-SDK][INFO] - Iteration 290, Energy -1.8318609326223656
[BQ-PYTHON-SDK][INFO] - Iteration 300, Energy -1.8318624882745604
[BQ-PYTHON-SDK][INFO] - Iteration 310, Energy -1.8318632688686038
[BQ-PYTHON-SDK][INFO] - Iteration 320, Energy -1.831863452938201
[BQ-PYTHON-SDK][INFO] - Iteration 330, Energy -1.831863494583107
[BQ-PYTHON-SDK]

## 3. Verify Against the Exact Solution

The qubit Hamiltonian is small (4 qubits, 15 terms), so we can classically diagonalize it with `NumPyMinimumEigensolver` to get the **exact** ground state energy. This serves as a reference to evaluate VQE accuracy.

The accepted ground state energy for H₂ in the STO-3G basis at this geometry is approximately **−1.8524 Hartree**. Chemical accuracy is typically defined as an error below **1.6 mHartree (≈ 1 kcal/mol)**.

In [4]:
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

# Classical exact diagonalization — feasible only for small Hamiltonians
solver = NumPyMinimumEigensolver()
exact_result = solver.compute_minimum_eigenvalue(qubit_op)
exact_energy = exact_result.eigenvalue.real

error_mhartree = abs(vqe_energy - exact_energy) * 1000

print(f"VQE energy:   {vqe_energy:.6f} Hartree")
print(f"Exact energy: {exact_energy:.6f} Hartree")
print(
    f"Error:        {error_mhartree:.2f} mHartree  ({abs(vqe_energy - exact_energy) / abs(exact_energy) * 100:.4f}%)"
)

VQE energy:   -1.831864 Hartree
Exact energy: -1.852388 Hartree
Error:        20.52 mHartree  (1.1080%)


## 4. Next Steps

- **Tune ansatz depth**: Try `options={"reps": 2}` or `options={"reps": 3}` — note that deeper ansätze have more parameters, which can make convergence slower.
- **Use BlueQubit cloud hardware**: Set `options={"estimator": "bluequbit"}` to run circuit evaluations on BlueQubit's quantum processors (requires a premium account).
- **Different molecules**: Replace the `PySCFDriver` geometry with another small molecule (LiH, BeH₂, etc.) — `hamiltonian_gs()` accepts any qubit Hamiltonian.